In [1]:
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader


c:\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
loaded_data=TextLoader("doc.txt")
docs=loaded_data.load()


text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50,separators=["\n\n","\n"," ",""])

chunks=text_splitter.split_documents(docs)

In [3]:
embeddings=OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore=FAISS.from_documents(chunks,embeddings)
vectorstore.similarity_search("How are autonomous AI agents being utilized in enterprise software development, and what are the main risks associated with their deployment?")

[Document(id='037044b4-1c24-4169-8e9d-1007019ca7f6', metadata={'source': 'doc.txt'}, page_content='- Software Development: Developers are increasingly integrating AI into their workflows. Tools powered by AI assist in code generation, bug detection, and automated testing. Complex architectures now frequently employ Retrieval-Augmented Generation (RAG) pipelines and multi-agent frameworks to build intelligent applications that can reason, search the web, and execute tools autonomously.'),
 Document(id='b75739b0-3663-4597-bd0f-ca9b98b8b409', metadata={'source': 'doc.txt'}, page_content='The Rise of Generative AI and Autonomous Agents'),
 Document(id='0bb046da-02ff-45db-bfba-9e6d0e762eb3', metadata={'source': 'doc.txt'}, page_content='Challenges and Ethical Considerations\nDespite its immense potential, the growing use of AI presents significant challenges. Data privacy, algorithmic bias, and the hallucination of facts in LLMs remain critical hurdles. As AI systems become more autonomous,

In [4]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# 1. Set your FAISS vectorstore to fetch a larger pool of documents first (e.g., top 10)
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# 2. Initialize a Cross-Encoder model (BGE is currently one of the best open-source options)
model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
compressor = CrossEncoderReranker(model=model, top_n=3) # Only keep the 3 best!

# 3. Wrap your base retriever with the re-ranker
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, 
    base_retriever=base_retriever
)

# 4. Run the query through the new pipeline!
query = "How are autonomous AI agents being utilized in enterprise software development, and what are the main risks associated with their deployment?"
reranked_docs = compression_retriever.invoke(query)

for i, doc in enumerate(reranked_docs):
    print(f"Rank {i+1}:\n{doc.page_content}\n")

c:\RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SAYANSH\.cache\huggingface\hub\models--BAAI--bge-reranker-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 201/201 [00:01<00:00, 119.12it/s, Materializing param=roberta.encoder.layer.11.output.dense.weigh

Rank 1:
Challenges and Ethical Considerations
Despite its immense potential, the growing use of AI presents significant challenges. Data privacy, algorithmic bias, and the hallucination of facts in LLMs remain critical hurdles. As AI systems become more autonomous, ensuring robust alignment with human values and establishing clear regulatory frameworks is paramount to mitigate risks associated with unchecked deployment.

Rank 2:
The most visible manifestation of AI's recent growth is Generative AI. Unlike traditional predictive models, generative systems can produce original text, images, and code. Furthermore, the ecosystem is evolving from simple prompt-response interactions to complex, stateful autonomous agents. These agents can break down complex objectives, formulate plans, and utilize external tools, drastically increasing the utility of AI in enterprise environments.

Rank 3:
- Software Development: Developers are increasingly integrating AI into their workflows. Tools powered 